In [ ]:
from google.colab import files
import os

# 1. Kaggle API kimlik dosyasını Colab'a yükle
print("Lütfen bilgisayarındaki kaggle.json dosyasını seçin:")
files.upload()

# 2. Dosyayı Linux ortamında olması gereken gizli klasöre taşı ve izinlerini ayarla
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. Bağlantıyı test et
print("\nKurulum Tamamlandı! Kaggle API kullanıma hazır.")

Lütfen bilgisayarındaki kaggle.json dosyasını seçin:


Saving kaggle.json to kaggle.json

Kurulum Tamamlandı! Kaggle API kullanıma hazır.


In [ ]:
# Veri setini Kaggle'dan indir
!kaggle datasets download -d mikhailma/test-dataset

# İnen zip dosyasını 'recaptcha_data' adında bir klasöre çıkart
!unzip -q test-dataset.zip -d recaptcha_data

Dataset URL: https://www.kaggle.com/datasets/mikhailma/test-dataset
License(s): CC0-1.0
100% 393M/393M [00:23<00:00, 17.6MB/s]



In [ ]:
# YOLOv8 için gerekli kütüphaneyi kur
!pip install ultralytics

# Kurulumun başarılı olduğunu ve GPU'nun aktif olduğunu test et
import ultralytics
ultralytics.checks()

Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.9/112.6 GB disk)


In [ ]:
!ls recaptcha_data/

Google_Recaptcha_V2_Images_Dataset


In [ ]:
# Klasörleri otomatik bölmek için gerekli kütüphaneyi kuruyoruz
!pip install split-folders

import splitfolders

# Sınıfların bulunduğu ana klasör yolu (images klasörü)
input_folder = "recaptcha_data/Google_Recaptcha_V2_Images_Dataset/images"

# YOLO'nun okuyacağı yeni formatın kaydedileceği klasör
output_folder = "yolo_recaptcha_dataset"

# Veriyi %80 Eğitim (train) ve %20 Doğrulama (val) olarak ayırıyoruz
splitfolders.ratio(input_folder, output=output_folder, seed=42, ratio=(.8, .2))

print("Veri seti başarıyla Train ve Val olarak ayrıldı!")

Copying files: 11730 files [00:03, 3168.44 files/s]

Veri seti başarıyla Train ve Val olarak ayrıldı!


In [ ]:
from ultralytics import YOLO

# YOLO11 Nano (En hafif 11. nesil model) yükleniyor
model = YOLO('yolo11n-cls.pt')

# Eğitim başlatılıyor (v8 ile aynı parametreler)
results = model.train(
    data='/content/yolo_recaptcha_dataset', # Veri seti yolu
    epochs=15,
    imgsz=128,
    batch=64,
    name='yolo11n_classification'
)

Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_recaptcha_dataset, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=128, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_classification, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_m

In [ ]:
from ultralytics import YOLO

# YOLO11 Small (v8s'in 11. nesil doğrudan rakibi)
model = YOLO('yolo11s-cls.pt')

# Eğitim başlatılıyor
results = model.train(
    data='/content/yolo_recaptcha_dataset',
    epochs=15,
    imgsz=128,
    batch=64,
    name='yolo11s_classification'
)

Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_recaptcha_dataset, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=128, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11s_classification, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_m

In [ ]:
from ultralytics import YOLO

# 1. Sınıflandırma için önceden eğitilmiş YOLOv8 Nano modelini yüklüyoruz
model = YOLO('yolov8n-cls.pt')

# 2. Model eğitimini başlatıyoruz
results = model.train(
    data='yolo_recaptcha_dataset', # Az önce oluşturduğumuz train/val klasör yolu
    epochs=15,                     # İlk etapta sistemi ve donanımı test etmek için 15 tur yeterli
    imgsz=128,                     # reCAPTCHA grid görselleri küçüktür, 128x128 boyut idealdir
    batch=64,                      # Colab'daki T4 GPU'sunun belleği 64 batch size'ı rahatça kaldırır
    project='Recaptcha_Solver',    # Sonuçların ve ağırlıkların kaydedileceği ana klasör
    name='yolov8_classification'   # Bu spesifik eğitimin adı
)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_recaptcha_dataset, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=128, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8_classification, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

In [ ]:
from ultralytics import YOLO

# Nano yerine daha fazla parametreye sahip olan Small (s) modelini yüklüyoruz
model = YOLO('yolov8s-cls.pt')

# Daha agresif parametrelerle eğitimi başlatıyoruz
results = model.train(
    data='yolo_recaptcha_dataset',
    epochs=50,                     # 15'ten 50'ye çıkardık
    patience=10,                   # Erken durdurma: 10 epoch boyunca doğruluk artmazsa eğitimi bitir
    imgsz=128,
    batch=64,
    project='Recaptcha_Solver',
    name='yolov8s_classification_boosted',
    # Veri Artırımı (Data Augmentation) Parametreleri
    degrees=10.0,                  # Görselleri rastgele max 10 derece döndür
    hsv_s=0.5,                     # reCAPTCHA görselleri genelde soluk veya kalitesizdir,
    hsv_v=0.5,                     # parlaklık ve doygunluk ile rastgele oynayarak modeli zorluyoruz.
    fliplr=0.5                     # Görsellerin yarısını yatayda çevir (aynalama)
)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_recaptcha_dataset, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.5, imgsz=128, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_classification_boosted, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_m

In [ ]:
import os
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# 1. Veri Yükleme ve HOG Özellik Çıkarımı Fonksiyonu
def load_and_extract_features(folder_path):
    features = []
    labels = []
    class_names = os.listdir(folder_path)

    for class_name in class_names:
        class_dir = os.path.join(folder_path, class_name)
        if not os.path.isdir(class_dir): continue

        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            # Görüntüyü gri tonlamalı oku ve 64x64'e yeniden boyutlandır (HOG için ideal)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None: continue
            img = cv2.resize(img, (64, 64))

            # HOG (Histogram of Oriented Gradients) Özellik Çıkarımı
            fd = hog(img, orientations=9, pixels_per_cell=(8, 8),
                     cells_per_block=(2, 2), visualize=False)

            features.append(fd)
            labels.append(class_name)

    return np.array(features), np.array(labels)

print("HOG özellikleri çıkarılıyor, bu işlem birkaç dakika sürebilir...")
X_train_hog, y_train_str = load_and_extract_features('yolo_recaptcha_dataset/train')
X_val_hog, y_val_str = load_and_extract_features('yolo_recaptcha_dataset/val')

# Etiketleri (Bisiklet, Otobüs vs.) sayısallaştırıyoruz (0, 1, 2...)
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)
y_val = le.transform(y_val_str)

print(f"HOG ile Çıkarılan Boyut: {X_train_hog.shape[1]}")

# 2. PCA ile Boyut İndirgeme (Feature Selection)
print("PCA uygulanıyor...")
# Özelliklerin %95 varyansını koruyacak şekilde boyutları indirge
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_hog)
X_val_pca = pca.transform(X_val_hog)

print(f"PCA Sonrası İndirgenen Boyut: {X_train_pca.shape[1]}")

# 3. SVM (Support Vector Machine) Modeli Eğitimi
print("SVM Modeli eğitiliyor...")
svm_model = SVC(kernel='rbf', gamma='scale')
svm_model.fit(X_train_pca, y_train)

# 4. Performans Ölçümü
y_pred = svm_model.predict(X_val_pca)
svm_accuracy = accuracy_score(y_val, y_pred)

print(f"\n--- MAKALEDE KULLANILACAK SONUÇLAR ---")
print(f"Geleneksel Yöntem (HOG + PCA + SVM) Başarısı: %{svm_accuracy * 100:.2f}")

HOG özellikleri çıkarılıyor, bu işlem birkaç dakika sürebilir...
HOG ile Çıkarılan Boyut: 1764
PCA uygulanıyor...
PCA Sonrası İndirgenen Boyut: 506
SVM Modeli eğitiliyor...

--- MAKALEDE KULLANILACAK SONUÇLAR ---
Geleneksel Yöntem (HOG + PCA + SVM) Başarısı: %48.11


In [ ]:
from google.colab import files
# Eğitilen en iyi model dosyasını bilgisayara indiriyoruz
files.download('/content/runs/classify/Recaptcha_Solver/yolov8_classification/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>